In [ ]:
import pandas as pd 
from scipy import stats
import import_ipynb
from scipy.stats import ttest_ind
from cuzick_test import cuzick_test
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats as stats
import scipy.io as sio
import import_ipynb
from benjamini_hochberg import benjamini_hochberg


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning) 

In [ ]:
def sig_star(p):
    star = ""
    if p <= 0.01:
        star = "***"
    elif p <= 0.05:
        star = "*"
    return star
    

In [ ]:
matched_df = pd.read_csv(r"matched_df.csv")

In [ ]:
def isNaN(num):
    return num != num
def compute_spec_power(freq,psd,lower_freq,upper_freq): 
    lower_freq_id = np.abs(freq-lower_freq).argmin()
    upper_freq_id = np.abs(freq-upper_freq).argmin()
    return np.nansum(psd[lower_freq_id:upper_freq_id])*(freq[1]-freq[0])
def get_weird_spec_indicator(spec, thres):
    # input: spec.shape = (# epoch, #psd, #channel)
    ind = np.any(np.any(np.abs(np.diff(np.diff(Wake_EEG_specs, axis=1),axis=1))>thres, axis=1),axis=1)
    return ind
def get_artifact_epoch(specs, freq, diff2_thres, freq_thres=4):
    freq_ids = freq>=freq_thres
    second_order_diff = np.diff(np.diff(specs[:,freq_ids],axis=1),axis=1)
    ids = np.any(np.any(np.abs(second_order_diff)>diff2_thres, axis=1), axis=1)
    return ids

In [ ]:
features_df = pd.read_csv("brain_age_features_list.csv")
features_df =matched_df[['FolderName','Predicted_Stage']].merge(features_df,on=['FolderName'])

# Get Data

In [ ]:
Wake_EEG_specs_all = np.empty((0,586,6))
N1_EEG_specs_all =np.empty((0,586,6))
N2_EEG_specs_all = np.empty((0,586,6))
N3_EEG_specs_all =np.empty((0,586,6))
REM_EEG_specs_all = np.empty((0,586,6))

for index,row in features_df[features_df['Predicted_Stage'] == 'Dementia'].iterrows():
    t = sio.loadmat(row['features_path'])
    freq = t['EEG_frequency'][0]

    Wake_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 5)]
    Wake_EEG_specs = Wake_EEG_specs[~get_artifact_epoch(Wake_EEG_specs, freq, 20, freq_thres=4)]
    Wake_EEG_specs = Wake_EEG_specs.mean(axis=0)

    N1_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 3)]
    N1_EEG_specs = N1_EEG_specs[~get_artifact_epoch(N1_EEG_specs, freq, 20, freq_thres=4)]
    N1_EEG_specs = N1_EEG_specs.mean(axis=0)
    
    N2_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 2)]
    N2_EEG_specs = N2_EEG_specs[~get_artifact_epoch(N2_EEG_specs, freq, 20, freq_thres=4)]
    N2_EEG_specs = N2_EEG_specs.mean(axis=0)
    
    N3_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 1)]
    N3_EEG_specs = N3_EEG_specs[~get_artifact_epoch(N3_EEG_specs, freq, 20, freq_thres=4)]
    N3_EEG_specs = N3_EEG_specs.mean(axis=0)
    
    REM_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 4)]
    REM_EEG_specs = REM_EEG_specs[~get_artifact_epoch(REM_EEG_specs, freq, 20, freq_thres=4)]
    REM_EEG_specs = REM_EEG_specs.mean(axis=0)
    
    if np.isnan(Wake_EEG_specs).any():
        continue
    if np.isnan(N1_EEG_specs).any():
        continue
    if np.isnan(N2_EEG_specs).any():
        continue
    if np.isnan(N3_EEG_specs).any():
        continue
    if np.isnan(REM_EEG_specs).any():
        continue
    
    Wake_EEG_specs_all = np.append(Wake_EEG_specs_all, Wake_EEG_specs.reshape(1,586,6),axis=0)
    N1_EEG_specs_all = np.append(N1_EEG_specs_all, N1_EEG_specs.reshape(1,586,6),axis=0)
    N2_EEG_specs_all = np.append(N2_EEG_specs_all, N2_EEG_specs.reshape(1,586,6),axis=0)
    N3_EEG_specs_all = np.append(N3_EEG_specs_all, N3_EEG_specs.reshape(1,586,6),axis=0)
    REM_EEG_specs_all = np.append(REM_EEG_specs_all, REM_EEG_specs.reshape(1,586,6),axis=0)
    
with open('Dementia_Wake_EEG_specs.npy', 'wb') as f:
    np.save(f, Wake_EEG_specs_all)
with open('Dementia_N1_EEG_specs.npy', 'wb') as f:
    np.save(f, N1_EEG_specs_all)
with open('Dementia_N2_EEG_specs.npy', 'wb') as f:
    np.save(f, N2_EEG_specs_all)
with open('Dementia_N3_EEG_specs.npy', 'wb') as f:
    np.save(f, N3_EEG_specs_all)
with open('Dementia_REM_EEG_specs.npy', 'wb') as f:
    np.save(f, REM_EEG_specs_all)

In [ ]:
Wake_EEG_specs_all = np.empty((0,586,6))
N1_EEG_specs_all =np.empty((0,586,6))
N2_EEG_specs_all = np.empty((0,586,6))
N3_EEG_specs_all =np.empty((0,586,6))
REM_EEG_specs_all = np.empty((0,586,6))

for index,row in features_df[features_df['Predicted_Stage'] == 'MCI'].iterrows():
    t = sio.loadmat(row['features_path'])
    freq = t['EEG_frequency'][0]

    Wake_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 5)]
    Wake_EEG_specs = Wake_EEG_specs[~get_artifact_epoch(Wake_EEG_specs, freq, 20, freq_thres=4)]
    Wake_EEG_specs = Wake_EEG_specs.mean(axis=0)

    N1_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 3)]
    N1_EEG_specs = N1_EEG_specs[~get_artifact_epoch(N1_EEG_specs, freq, 20, freq_thres=4)]
    N1_EEG_specs = N1_EEG_specs.mean(axis=0)
    
    N2_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 2)]
    N2_EEG_specs = N2_EEG_specs[~get_artifact_epoch(N2_EEG_specs, freq, 20, freq_thres=4)]
    N2_EEG_specs = N2_EEG_specs.mean(axis=0)
    
    N3_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 1)]
    N3_EEG_specs = N3_EEG_specs[~get_artifact_epoch(N3_EEG_specs, freq, 20, freq_thres=4)]
    N3_EEG_specs = N3_EEG_specs.mean(axis=0)
    
    REM_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 4)]
    REM_EEG_specs = REM_EEG_specs[~get_artifact_epoch(REM_EEG_specs, freq, 20, freq_thres=4)]
    REM_EEG_specs = REM_EEG_specs.mean(axis=0)
    
    if np.isnan(Wake_EEG_specs).any():
        continue
    if np.isnan(N1_EEG_specs).any():
        continue
    if np.isnan(N2_EEG_specs).any():
        continue
    if np.isnan(N3_EEG_specs).any():
        continue
    if np.isnan(REM_EEG_specs).any():
        continue
    
    Wake_EEG_specs_all = np.append(Wake_EEG_specs_all, Wake_EEG_specs.reshape(1,586,6),axis=0)
    N1_EEG_specs_all = np.append(N1_EEG_specs_all, N1_EEG_specs.reshape(1,586,6),axis=0)
    N2_EEG_specs_all = np.append(N2_EEG_specs_all, N2_EEG_specs.reshape(1,586,6),axis=0)
    N3_EEG_specs_all = np.append(N3_EEG_specs_all, N3_EEG_specs.reshape(1,586,6),axis=0)
    REM_EEG_specs_all = np.append(REM_EEG_specs_all, REM_EEG_specs.reshape(1,586,6),axis=0)
    
with open('MCI_Wake_EEG_specs.npy', 'wb') as f:
    np.save(f, Wake_EEG_specs_all)
with open('MCI_N1_EEG_specs.npy', 'wb') as f:
    np.save(f, N1_EEG_specs_all)
with open('MCI_N2_EEG_specs.npy', 'wb') as f:
    np.save(f, N2_EEG_specs_all)
with open('MCI_N3_EEG_specs.npy', 'wb') as f:
    np.save(f, N3_EEG_specs_all)
with open('MCI_REM_EEG_specs.npy', 'wb') as f:
    np.save(f, REM_EEG_specs_all)

In [ ]:
Wake_EEG_specs_all = np.empty((0,586,6))
N1_EEG_specs_all =np.empty((0,586,6))
N2_EEG_specs_all = np.empty((0,586,6))
N3_EEG_specs_all =np.empty((0,586,6))
REM_EEG_specs_all = np.empty((0,586,6))

for index,row in features_df[features_df['Predicted_Stage'] == 'No Dementia'].iterrows():
    t = sio.loadmat(row['features_path'])
    freq = t['EEG_frequency'][0]

    Wake_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 5)]
    Wake_EEG_specs = Wake_EEG_specs[~get_artifact_epoch(Wake_EEG_specs, freq, 20, freq_thres=4)]
    Wake_EEG_specs = Wake_EEG_specs.mean(axis=0)

    N1_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 3)]
    N1_EEG_specs = N1_EEG_specs[~get_artifact_epoch(N1_EEG_specs, freq, 20, freq_thres=4)]
    N1_EEG_specs = N1_EEG_specs.mean(axis=0)
    
    N2_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 2)]
    N2_EEG_specs = N2_EEG_specs[~get_artifact_epoch(N2_EEG_specs, freq, 20, freq_thres=4)]
    N2_EEG_specs = N2_EEG_specs.mean(axis=0)
    
    N3_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 1)]
    N3_EEG_specs = N3_EEG_specs[~get_artifact_epoch(N3_EEG_specs, freq, 20, freq_thres=4)]
    N3_EEG_specs = N3_EEG_specs.mean(axis=0)
    
    REM_EEG_specs = t['EEG_specs'][np.where(t['sleep_stages'][0] == 4)]
    REM_EEG_specs = REM_EEG_specs[~get_artifact_epoch(REM_EEG_specs, freq, 20, freq_thres=4)]
    REM_EEG_specs = REM_EEG_specs.mean(axis=0)
    
    if np.isnan(Wake_EEG_specs).any():
        continue
    if np.isnan(N1_EEG_specs).any():
        continue
    if np.isnan(N2_EEG_specs).any():
        continue
    if np.isnan(N3_EEG_specs).any():
        continue
    if np.isnan(REM_EEG_specs).any():
        continue
    
    Wake_EEG_specs_all = np.append(Wake_EEG_specs_all, Wake_EEG_specs.reshape(1,586,6),axis=0)
    N1_EEG_specs_all = np.append(N1_EEG_specs_all, N1_EEG_specs.reshape(1,586,6),axis=0)
    N2_EEG_specs_all = np.append(N2_EEG_specs_all, N2_EEG_specs.reshape(1,586,6),axis=0)
    N3_EEG_specs_all = np.append(N3_EEG_specs_all, N3_EEG_specs.reshape(1,586,6),axis=0)
    REM_EEG_specs_all = np.append(REM_EEG_specs_all, REM_EEG_specs.reshape(1,586,6),axis=0)
    
with open('CN_Wake_EEG_specs.npy', 'wb') as f:
    np.save(f, Wake_EEG_specs_all)
with open('CN_N1_EEG_specs.npy', 'wb') as f:
    np.save(f, N1_EEG_specs_all)
with open('CN_N2_EEG_specs.npy', 'wb') as f:
    np.save(f, N2_EEG_specs_all)
with open('CN_N3_EEG_specs.npy', 'wb') as f:
    np.save(f, N3_EEG_specs_all)
with open('CN_REM_EEG_specs.npy', 'wb') as f:
    np.save(f, REM_EEG_specs_all)

# Spectral Analysis

In [ ]:
def get_peaks(freq,psd):
    spec_df = pd.DataFrame(columns=['Frequency','PSD'])
    spec_df['Frequency'] = freq
    spec_df['PSD'] = psd

    alpha_spec_df = spec_df[(spec_df['Frequency'] >= 5) & (spec_df['Frequency'] <= 15)]
    IAF_peak = alpha_spec_df.sort_values(by=['PSD'],ascending=False)['Frequency'].iloc[0]
    if len(alpha_spec_df[alpha_spec_df['Frequency'] < IAF_peak].sort_values(by=['PSD'])['Frequency']) == 0:
        theta_spec_df = spec_df[(spec_df['Frequency'] >= 4) & (spec_df['Frequency'] <= 7)]
        TF = theta_spec_df.sort_values(by=['PSD'])['Frequency'].iloc[0]
        alpha_spec_df = spec_df[(spec_df['Frequency'] >= TF) & (spec_df['Frequency'] <= 15)]
        IAF_peak = alpha_spec_df.sort_values(by=['PSD'],ascending=False)['Frequency'].iloc[0]
    else:
        TF = alpha_spec_df[alpha_spec_df['Frequency'] < IAF_peak].sort_values(by=['PSD'])['Frequency'].iloc[0]
    
    return TF,IAF_peak

In [ ]:
def get_spindle_peak(freq,psd):
    spec_df = pd.DataFrame(columns=['Frequency','PSD'])
    spec_df['Frequency'] = freq
    spec_df['PSD'] = psd
    spindle_spec_df = spec_df[(spec_df['Frequency'] >= 10) & (spec_df['Frequency'] <= 15)]
    spindle_spec_df['PSD_diff'] = np.append(np.array([np.nan]) ,(np.diff(spindle_spec_df['PSD'])))
    
    if len(((np.diff(np.sign(np.diff(spindle_spec_df['PSD']))) <0 ).nonzero()[0] + 1)) == 0:
        return 13
        #there is no spindle peak default to 13 Hz
    n = ((np.diff(np.sign(np.diff(spindle_spec_df['PSD']))) <0 ).nonzero()[0] + 1)[-1]
    spindle_peak = spindle_spec_df.iloc[n]['Frequency']
    
    return spindle_peak

In [ ]:
#RMSD with predicted value being mean
def calculate_medoid(psd_array):
    mean_array = np.mean(psd_array,axis=0)
    D_array = np.sqrt(np.sum((mean_array-psd_array)**2,axis=1)/psd_array.shape[1])
    min_index = np.argmin(D_array)
    return psd_array[min_index]

In [ ]:
def compute_CI(psd_array):
    mean_array = np.mean(psd_array,axis=0)
    ci_array = 1.96*stats.sem(psd_array)
    return mean_array + ci_array, mean_array - ci_array

In [ ]:
freq = np.load('freq.npy')

In [ ]:
p_list = []
CN_spec = np.load('CN_Wake_EEG_specs.npy')
CN_spec = (CN_spec[:,:,4] + CN_spec[:,:,5])/2
MCI_spec = np.load('MCI_Wake_EEG_specs.npy')
MCI_spec = (MCI_spec[:,:,4] + MCI_spec[:,:,5])/2
DM_spec = np.load('Dementia_Wake_EEG_specs.npy')
DM_spec = (DM_spec[:,:,4] + DM_spec[:,:,5])/2
for f in range(len(freq)):
    s,p = stats.f_oneway(CN_spec[:,f],MCI_spec[:,f],DM_spec[:,f])
    p_list.append(p)
cutoff=0.05
sig_list = np.array([1 if p <cutoff else np.nan for p in p_list])

In [ ]:
sig_list

In [ ]:
fig, ax1 = plt.subplots(1,1 ,figsize=(11,7),dpi=300)



ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('dB')
ax1.set_title('Wake Occipital')
ax1.set_xlim([1,20])



W_IAF_array = []
W_TF_array = []
N1_IAF_array = []
N1_TF_array = []
N2_Spindle_array = []
N3_Spindle_array = []
REM_IAF_array = []
REM_TF_array = []

stages = ['CN','MCI','Dementia']
colors = ['tab:blue','tab:orange','tab:red']
for i in range(3):
    Wake_EEG_specs_All = np.load(stages[i] +'_Wake_EEG_specs.npy')
    spec = (Wake_EEG_specs_All[:,:,4] + Wake_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    TF,IAF_peak = get_peaks(freq,mean_array)


    upper,lower = compute_CI(spec)
    ax1.plot(freq,mean_array,color=colors[i],zorder=3)
    ax1.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)
    

    ax1.scatter(IAF_peak,mean_array[np.where(freq==IAF_peak)[0][0]],marker='o',color=colors[i],s=20,zorder=4)
    x=IAF_peak
    y = mean_array[np.where(freq==IAF_peak)[0][0]]
    ax1.plot([x,x],[-10,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax1.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    W_IAF_array.append('(' + str(round(x,2)) + ',' + str(round(y,2)) + ')')
    
    ax1.scatter(TF,mean_array[np.where(freq==TF)[0][0]],marker='^',color=colors[i],s=20,zorder=4)
    x=TF
    y = mean_array[np.where(freq==TF)[0][0]]
    ax1.plot([x,x],[-10,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax1.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    W_TF_array.append('(' + str(round(x,2)) + ',' + str(round(y,2)) + ')')

    N1_EEG_specs_All = np.load(stages[i] +'_N1_EEG_specs.npy')
    spec = (N1_EEG_specs_All[:,:,4] + N1_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    TF,IAF_peak = get_peaks(freq,mean_array)

    


ax1.set_ylim([-10, 40])

from matplotlib.lines import Line2D
custom_labels = [Line2D([0], [0], color='tab:red', lw=4),
                Line2D([0], [0], color='tab:orange', lw=4),
                Line2D([0], [0], color='tab:blue', lw=4),
                Line2D([0], [0], marker='o', color='w', label='Scatter', markerfacecolor='black', markersize=9),
                Line2D([0], [0], marker='^', color='w', label='Scatter', markerfacecolor='black', markersize=10),
                Line2D([0], [0], marker='*', color='w', label='Scatter', markerfacecolor='black', markersize=14)]
#fig.delaxes(_)
#_.axis('off')
#_.legend(custom_labels, ['DEM', 'MCI', 'CN','IAF','TF','Spindle'],ncol=2,frameon=False,loc='center')


y_shift = 0

for i in range(3):
    if i == 0:
        lw=0.7
    else:
        lw=0
    ax1.annotate(W_IAF_array[i],
                xy=(8.5,20.5), xycoords='data',
                xytext=(40, 0+y_shift), textcoords='offset points',color=colors[i],fontsize=10,arrowprops=dict(arrowstyle='->',lw=lw))
    y_shift+=12


        
    
fig.tight_layout()
fig.subplots_adjust(hspace=0.4)
#fig.subplots_adjust(vspace=.3)
#plt.savefig(r'..\figures\Fig3.svg',format='svg',bbox_inches=0)
plt.show()

            
            

In [ ]:
fig, ((ax1,ax2,_),(ax3,ax4,ax5)) = plt.subplots(2,3 ,figsize=(11,7),dpi=300)



ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('dB')
ax1.set_title('Wake Occipital')
ax1.set_xlim([1,20])

ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('dB')
ax2.set_title('N1 Occipital')
ax2.set_xlim([1,20])

ax3.set_xlabel('Frequency (Hz)')
ax3.set_ylabel('dB')
ax3.set_title('N2 Occipital')
ax3.set_xlim([1,20])

ax4.set_xlabel('Frequency (Hz)')
ax4.set_ylabel('dB')
ax4.set_title('N3 Occipital')
ax4.set_xlim([1,20])

ax5.set_xlabel('Frequency (Hz)')
ax5.set_ylabel('dB')
ax5.set_title('REM Occipital')
ax5.set_xlim([1,20])

W_IAF_array = []
W_TF_array = []
N1_IAF_array = []
N1_TF_array = []
N2_Spindle_array = []
N3_Spindle_array = []
REM_IAF_array = []
REM_TF_array = []

stages = ['CN','MCI','Dementia']
colors = ['tab:blue','tab:orange','tab:red']
for i in range(3):
    Wake_EEG_specs_All = np.load(stages[i] +'_Wake_EEG_specs.npy')
    spec = (Wake_EEG_specs_All[:,:,4] + Wake_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    TF,IAF_peak = get_peaks(freq,mean_array)


    upper,lower = compute_CI(spec)
    ax1.plot(freq,mean_array,color=colors[i],zorder=3)
    ax1.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)
    

    ax1.scatter(IAF_peak,mean_array[np.where(freq==IAF_peak)[0][0]],marker='o',color=colors[i],s=20,zorder=4)
    x=IAF_peak
    y = mean_array[np.where(freq==IAF_peak)[0][0]]
    ax1.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax1.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    W_IAF_array.append(r'$\bf{' + str(round(x,2)) + '}$,' + str(round(y,2)) )
    
    ax1.scatter(TF,mean_array[np.where(freq==TF)[0][0]],marker='^',color=colors[i],s=20,zorder=4)
    x=TF
    y = mean_array[np.where(freq==TF)[0][0]]
    ax1.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax1.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    W_TF_array.append( str(round(x,2)) + r',$\bf{' + str(round(y,2)) + '}$')

    N1_EEG_specs_All = np.load(stages[i] +'_N1_EEG_specs.npy')
    spec = (N1_EEG_specs_All[:,:,4] + N1_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    TF,IAF_peak = get_peaks(freq,mean_array)

    upper,lower = compute_CI(spec)
    ax2.plot(freq,mean_array,color=colors[i],zorder=3,label=stages[i])
    ax2.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)

    ax2.scatter(IAF_peak,mean_array[np.where(freq==IAF_peak)[0][0]],marker='o',color=colors[i],s=20,zorder=4)
    x=IAF_peak
    y = mean_array[np.where(freq==IAF_peak)[0][0]]
    ax2.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax2.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    N1_IAF_array.append(r'$\bf{' + str(round(x,2)) + r'}$,' + str(round(y,2)) )
    
    ax2.scatter(TF,mean_array[np.where(freq==TF)[0][0]],marker='^',color=colors[i],s=20,zorder=4)
    x=TF
    y = mean_array[np.where(freq==TF)[0][0]]
    ax2.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax2.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    N1_TF_array.append(r'$\bf{' + str(round(x,2)) + r'}$,$\bf{' + str(round(y,2)) + '}$')
    
    N2_EEG_specs_All = np.load(stages[i] +'_N2_EEG_specs.npy')
    spec = (N2_EEG_specs_All[:,:,4] + N2_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    spindle_peak = get_spindle_peak(freq,mean_array)

    upper,lower = compute_CI(spec)
    ax3.plot(freq,mean_array,color=colors[i])
    ax3.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)
    x=spindle_peak
    y = mean_array[np.where(freq==spindle_peak)[0][0]]
    ax3.scatter(x,y,marker='*',color=colors[i],s=20,zorder=4)
    ax3.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax3.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    N2_Spindle_array.append(str(round(x,2)) + r',$\bf{' + str(round(y,2)) + '}$')
    
    N3_EEG_specs_All = np.load(stages[i] +'_N3_EEG_specs.npy')
    spec = (N3_EEG_specs_All[:,:,4] + N3_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    spindle_peak = get_spindle_peak(freq,mean_array)

    upper,lower = compute_CI(spec)
    ax4.plot(freq,mean_array,color=colors[i])
    ax4.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)
    x=spindle_peak
    y = mean_array[np.where(freq==spindle_peak)[0][0]]
    ax4.scatter(x,y,marker='*',color=colors[i],s=20,zorder=4)
    ax4.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax4.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    N3_Spindle_array.append(str(round(x,2)) + ',' + str(round(y,2)) )

    REM_EEG_specs_All = np.load(stages[i] +'_REM_EEG_specs.npy')
    spec = (REM_EEG_specs_All[:,:,4] + REM_EEG_specs_All[:,:,5])/2
    spec = 10*np.log(spec)
    mean_array = spec.mean(axis=0)  
    TF,IAF_peak = get_peaks(freq,mean_array)

    upper,lower = compute_CI(spec)
    ax5.plot(freq,mean_array,color=colors[i],zorder=3)
    ax5.fill_between(freq, lower, upper, color=colors[i], alpha=.3,zorder=3)
    ax5.scatter(IAF_peak,mean_array[np.where(freq==IAF_peak)[0][0]],marker='o',color=colors[i],s=20,zorder=4)
    x=IAF_peak
    y = mean_array[np.where(freq==IAF_peak)[0][0]]
    ax5.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax5.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    REM_IAF_array.append(r'$\bf{' + str(round(x,2)) + '}$,' + str(round(y,2)) )
    
    ax5.scatter(TF,mean_array[np.where(freq==TF)[0][0]],marker='^',color=colors[i],s=20,zorder=4)
    x=TF
    y = mean_array[np.where(freq==TF)[0][0]]
    ax5.plot([x,x],[-15,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    ax5.plot([freq[0],x],[y,y],color=colors[i],linestyle='--',linewidth=1,zorder=3)
    REM_TF_array.append( str(round(x,2)) + r',$\bf{' + str(round(y,2)) + '}$')


ax1.set_ylim([-15, 40])
ax2.set_ylim([-15, 40])
ax3.set_ylim([-15, 40])
ax4.set_ylim([-15, 40])
ax5.set_ylim([-15, 40])
from matplotlib.lines import Line2D
custom_labels = [Line2D([0], [0], color='tab:red', lw=4),
                Line2D([0], [0], color='tab:orange', lw=4),
                Line2D([0], [0], color='tab:blue', lw=4),
                Line2D([0], [0], marker='o', color='w', label='Scatter', markerfacecolor='black', markersize=9),
                Line2D([0], [0], marker='^', color='w', label='Scatter', markerfacecolor='black', markersize=10),
                Line2D([0], [0], marker='*', color='w', label='Scatter', markerfacecolor='black', markersize=14)]
#fig.delaxes(_)
_.axis('off')
_.legend(custom_labels, ['DEM', 'MCI', 'CN','IAF','TF','Spindle'],ncol=2,frameon=False,loc='center')


y_shift = 0

for i in range(3):
    if i == 0:
        lw=0.7
    else:
        lw=0
    ax1.annotate(W_IAF_array[i],
                xy=(8.5,20.5), xycoords='data',
                xytext=(40, 0+y_shift), textcoords='offset points',color=colors[i],fontsize=10,arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax1.annotate(W_TF_array[i],
                xy=(5, 13), xycoords='data',
                xytext=(-27, 45+y_shift), textcoords='offset points',color=colors[i],arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax2.annotate(N1_IAF_array[i],
                xy=(8,18), xycoords='data',
                xytext=(40, 10+y_shift), textcoords='offset points',color=colors[i],fontsize=10,arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax2.annotate(N1_TF_array[i],
                xy=(5, 16), xycoords='data',
                xytext=(-30, 35+y_shift), textcoords='offset points',color=colors[i],arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax3.annotate(N2_Spindle_array[i],
                xy=(13.1, 3), xycoords='data',
                xytext=(-31, 35+y_shift), textcoords='offset points',color=colors[i],arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax4.annotate(N3_Spindle_array[i],
                xy=(13.2, 1), xycoords='data',
                xytext=(-33, 35+y_shift), textcoords='offset points',color=colors[i],arrowprops=dict(arrowstyle='->',lw=lw))

    ax5.annotate(REM_IAF_array[i],
                xy=(7.5,12), xycoords='data',
                xytext=(40, 10+y_shift), textcoords='offset points',color=colors[i],fontsize=10,arrowprops=dict(arrowstyle='->',lw=lw))
    
    ax5.annotate(REM_TF_array[i],
                xy=(6, 12), xycoords='data',
                xytext=(-24, 35+y_shift), textcoords='offset points',color=colors[i],arrowprops=dict(arrowstyle='->',lw=lw))

    y_shift+=12
sleep_stages=['Wake','N1','N2','N3','REM']
axes = [ax1,ax2,ax3,ax4,ax5]
for i in range(5):
    p_list = []
    CN_spec = np.load('CN_'+ sleep_stages[i] + '_EEG_specs.npy')
    CN_spec = (CN_spec[:,:,4] + CN_spec[:,:,5])/2
    CN_spec = 10*np.log(CN_spec)
    MCI_spec = np.load('MCI_'+ sleep_stages[i] +'_EEG_specs.npy')
    MCI_spec = (MCI_spec[:,:,4] + MCI_spec[:,:,5])/2
    MCI_spec = 10*np.log(MCI_spec)
    DM_spec = np.load('Dementia_'+ sleep_stages[i] +'_EEG_specs.npy')
    DM_spec = (DM_spec[:,:,4] + DM_spec[:,:,5])/2
    DM_spec = 10*np.log(DM_spec)
    for f in range(len(freq)):
        s,p = stats.f_oneway(CN_spec[:,f],MCI_spec[:,f],DM_spec[:,f])
        p_list.append(p)
    #p_vals_below, cutoff = benjamini_hochberg(np.array(p_list),0.05)
    cutoff = 0.05
    sig_list = np.array([1 if p <cutoff else np.nan for p in p_list])
    axes[i].plot(freq, sig_list*(-10),linewidth = 3,color='black',solid_capstyle='round');
    #x1=freq[0]
    #s1 = sig_list[0]
    #for f in range(len(freq)):
    #    if f == 585:
    #        break
    #    x2 = freq[f+1]
    #    s2 = sig_list[f+1]
    #    if s1 != s2:
    #        if s1 == 1:
    #            axes[i].fill_between([x1,freq[f]],-15,40, color='lightgrey', alpha=0.4)
    #        x1 = x2 
    #        s1 = s2
    
fig.tight_layout()
fig.subplots_adjust(hspace=0.4)
#fig.subplots_adjust(vspace=.3)

plt.savefig(r'../figures/Fig3.svg',format='svg',bbox_inches=0)
plt.show()

            
            

# ANOVA and post-hoc Tests

In [ ]:
#IAF #TF and Spindle
sleep_stages =['Wake','N1','REM']
for i in range(3):
    print('\n')
    print(sleep_stages[i])
    CN_spec = np.load('CN_'+sleep_stages[i]+'_EEG_specs.npy')
    CN_spec = (CN_spec[:,:,4] + CN_spec[:,:,5])/2
    CN_spec = 10*np.log(CN_spec)
    MCI_spec = np.load('MCI_'+sleep_stages[i]+'_EEG_specs.npy')
    MCI_spec = (MCI_spec[:,:,4] + MCI_spec[:,:,5])/2
    MCI_spec = 10*np.log(MCI_spec)
    DM_spec = np.load('Dementia_'+sleep_stages[i]+'_EEG_specs.npy')
    DM_spec = (DM_spec[:,:,4] + DM_spec[:,:,5])/2
    DM_spec = 10*np.log(DM_spec)
    #mean_array = spec.mean(axis=0) 
    DM_TF_list = []
    DM_IAF_list = []
    for i in range(len(DM_spec)):
        spec= DM_spec[i]
        TF,IAF_peak = get_peaks(freq,spec)
        DM_TF_list.append((TF,spec[np.where(freq==TF)[0][0]]))
        DM_IAF_list.append((IAF_peak,spec[np.where(freq==IAF_peak)[0][0]]))

    MCI_TF_list = []
    MCI_IAF_list = []
    for i in range(len(MCI_spec)):
        spec= MCI_spec[i]
        TF,IAF_peak = get_peaks(freq,spec)
        MCI_TF_list.append((TF,spec[np.where(freq==TF)[0][0]]))
        MCI_IAF_list.append((IAF_peak,spec[np.where(freq==IAF_peak)[0][0]]))

    CN_TF_list = []
    CN_IAF_list = []
    for i in range(len(CN_spec)):
        spec= CN_spec[i]
        TF,IAF_peak = get_peaks(freq,spec)
        CN_TF_list.append((TF,spec[np.where(freq==TF)[0][0]]))
        CN_IAF_list.append((IAF_peak,spec[np.where(freq==IAF_peak)[0][0]]))

    print('TF (Frequency)')
    s,p = stats.f_oneway([t[0] for t in CN_TF_list],[t[0] for t in MCI_TF_list],[t[0] for t in DM_TF_list])
    print('ANOVA p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_TF_list],[t[0] for t in CN_TF_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in CN_TF_list],[t[0] for t in MCI_TF_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_TF_list],[t[0] for t in MCI_TF_list])  
    print('DM vs MCI p:',p)
    
    print('TF (Power)')
    s,p = stats.f_oneway([t[1] for t in CN_TF_list],[t[1] for t in MCI_TF_list],[t[1] for t in DM_TF_list])
    print('ANOVA p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_TF_list],[t[1] for t in CN_TF_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in CN_TF_list],[t[1] for t in MCI_TF_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_TF_list],[t[1] for t in MCI_TF_list])  
    print('DM vs MCI p:',p)
    
    print('IAF (Frequency)')
    s,p = stats.f_oneway([t[0] for t in CN_IAF_list],[t[0] for t in MCI_IAF_list],[t[0] for t in DM_IAF_list])
    print('ANOVA p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_IAF_list],[t[0] for t in CN_IAF_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in CN_IAF_list],[t[0] for t in MCI_IAF_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_IAF_list],[t[0] for t in MCI_IAF_list])  
    print('DM vs MCI p:',p)
    
    print('IAF (Power)')
    s,p = stats.f_oneway([t[1] for t in CN_IAF_list],[t[1] for t in MCI_IAF_list],[t[1] for t in DM_IAF_list])
    print('ANVOA p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_IAF_list],[t[1] for t in CN_IAF_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in CN_IAF_list],[t[1] for t in MCI_IAF_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_IAF_list],[t[1] for t in MCI_IAF_list])  
    print('DM vs MCI p:',p)

In [ ]:
#Spindle
sleep_stages =['N2','N3']
for i in range(2):
    print('\n')
    print(sleep_stages[i])
    CN_spec = np.load('CN_'+sleep_stages[i]+'_EEG_specs.npy')
    CN_spec = (CN_spec[:,:,4] + CN_spec[:,:,5])/2
    CN_spec = 10*np.log(CN_spec)
    MCI_spec = np.load('MCI_'+sleep_stages[i]+'_EEG_specs.npy')
    MCI_spec = (MCI_spec[:,:,4] + MCI_spec[:,:,5])/2
    MCI_spec = 10*np.log(MCI_spec)
    DM_spec = np.load('Dementia_'+sleep_stages[i]+'_EEG_specs.npy')
    DM_spec = (DM_spec[:,:,4] + DM_spec[:,:,5])/2
    DM_spec = 10*np.log(DM_spec)
    #mean_array = spec.mean(axis=0) 
    DM_spindle_list = []
    for i in range(len(DM_spec)):
        spec= DM_spec[i]
        spindle_peak = get_spindle_peak(freq,spec)
        DM_spindle_list.append((spindle_peak,spec[np.where(freq==spindle_peak)[0][0]]))
       
    MCI_spindle_list = []
    for i in range(len(MCI_spec)):
        spec= MCI_spec[i]
        spindle_peak = get_spindle_peak(freq,spec)
        MCI_spindle_list.append((spindle_peak,spec[np.where(freq==spindle_peak)[0][0]]))

    CN_spindle_list = []
    for i in range(len(CN_spec)):
        spec= CN_spec[i]
        spindle_peak = get_spindle_peak(freq,spec)
        CN_spindle_list.append((spindle_peak,spec[np.where(freq==spindle_peak)[0][0]]))
    
    print('Spindle (Frequency)')
    s,p = stats.f_oneway([t[0] for t in CN_spindle_list],[t[0] for t in MCI_spindle_list],[t[0] for t in DM_spindle_list])
    print('ANOVA p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_spindle_list],[t[0] for t in CN_spindle_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in CN_spindle_list],[t[0] for t in MCI_spindle_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[0] for t in DM_spindle_list],[t[0] for t in MCI_spindle_list])  
    print('DM vs MCI p:',p)
    
    print('Spindle (Power)')
    s,p = stats.f_oneway([t[1] for t in CN_spindle_list],[t[1] for t in MCI_spindle_list],[t[1] for t in DM_spindle_list])
    print('ANOVA p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_spindle_list],[t[1] for t in CN_spindle_list])  
    print('DM vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in CN_spindle_list],[t[1] for t in MCI_spindle_list])  
    print('MCI vs CN p:',p)
    t,p =stats.ttest_ind([t[1] for t in DM_spindle_list],[t[1] for t in MCI_spindle_list])  
    print('DM vs MCI p:',p)

  